# 04 · Camada Silver — Continuidade

## 1. Objetivo e método

Esta etapa transforma a Bronze de continuidade no modelo que a Gold consome. A Bronze preserva o dado como a ANEEL publicou; a Silver corrige o que o notebook 03 apontou e entrega uma base tipada, deduplicada e com os indicadores compostos.

### O que a Silver resolve

Os achados do notebook 03 viram tratamento aqui, um a um:

| Achado da Bronze | Tratamento na Silver |
|---|---|
| `NumCNPJ` e `IdeConjUndConsumidoras` publicados como inteiro | Conversão para texto de 14 e 5 caracteres, com zeros à esquerda |
| Quatro chaves com linhas idênticas duplicadas | `dropDuplicates` sobre a chave completa |
| Formato longo, uma linha por parcela | Pivot para uma coluna por parcela |
| Composição com dois regimes normativos | Regra condicionada ao ano, conforme o PRODIST Módulo 8 |
| Parcela `INC` ausente em 44 conjuntos da EMR | Preenchimento com zero, por só ser informada em Dia Crítico |
| Ano civil incompleto em algumas distribuidoras | Flags `meses_no_ano` e `ano_completo`, sem excluir linha |

### O que a Silver não faz

Não filtra janela nem universo. As duas coisas são recorte analítico e pertencem à Gold. A Silver guarda a série inteira publicada e registra o porte da distribuidora como atributo, para que o critério fique audítavel no dado em vez de escondido dentro de um filtro.

### Tabelas produzidas

| Tabela | Grão | Papel |
|---|---|---|
| `fato_continuidade_mensal` | conjunto, ano, mês | Parcelas, peso e indicadores compostos |
| `dim_conjunto` | conjunto | Identidade e período de existência na série |
| `dim_distribuidora` | CNPJ | Sigla, porte em UCs e flag de grande porte |
| `recon_continuidade` | conjunto, ano, mês, indicador | Composto contra o publicado, para conferência |


## 2. Configuração

In [ ]:
import os
import sys

from pyspark.sql import functions as F
from pyspark.sql import Window

REPO_ROOT = os.path.dirname(os.getcwd())
if REPO_ROOT not in sys.path:
    sys.path.append(REPO_ROOT)

from src.config import CATALOG, SCHEMA_BRONZE, SCHEMA_SILVER

BRONZE = f"{CATALOG}.{SCHEMA_BRONZE}"
SILVER = f"{CATALOG}.{SCHEMA_SILVER}"

# Analysis window, kept here only for the checks at the end: the Silver tables
# themselves hold the whole published series.
ANO_INICIAL = 2022
ANO_FINAL = 2025
ANOS_JANELA = list(range(ANO_INICIAL, ANO_FINAL + 1))

# Reference year for the size criterion: the first year of the window.
ANO_REFERENCIA_PORTE = ANO_INICIAL
CORTE_UCS = 400_000

# Rounding tolerance accepted when comparing against ANEEL published aggregates
TOLERANCIA = 0.05

# Normative composition of DEC and FEC, PRODIST Module 8, Section 8.2.
# External parcels left the composition from 2022 onwards.
COMPOSICAO = {
    "ate_2021": ["IND", "IP", "XN", "XP"],
    "desde_2022": ["IND", "IP"],
}
ANO_MUDANCA_REGIME = 2022

# Own indicator: internal origin, unplanned, including ISE and Critical Day
PARCELAS_FI = ["IND", "INE", "INC"]

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER}")
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA_SILVER}")

print(f"Origem..........: {BRONZE}")
print(f"Destino.........: {SILVER}")
print(f"Janela..........: {ANO_INICIAL} a {ANO_FINAL} (anos civis)")
print(f"Porte...........: NumCon de dezembro de {ANO_REFERENCIA_PORTE}, corte em {CORTE_UCS:,} UCs")

## 3. Normalização a partir da Bronze

Três passos, na ordem em que um depende do anterior: tipar os identificadores, remover as duplicatas e girar o formato longo para largo. Só depois disso os indicadores podem ser compostos.

### 3.1 Tipagem e deduplicação

`NumCNPJ` e `IdeConjUndConsumidoras` chegam como inteiro e perdem os zeros à esquerda: 3.068.970 linhas têm CNPJ de 13 dígitos e 8.193 têm código de conjunto abaixo de cinco. Qualquer junção por esses campos falharia silenciosamente, por isso a conversão vem antes de tudo.

A deduplicação remove as quatro chaves que a ANEEL publicou em duplicidade exata, todas na ELEKTRO em junho de 2025. Como o notebook 03 confirmou que nenhuma delas carrega valor divergente, descartar a repetição é seguro e não exige critério de escolha.

In [ ]:
CHAVE = ["num_cnpj", "ide_conjunto", "ano", "mes", "sig_indicador"]

bronze = spark.table(f"{BRONZE}.continuity_indicators")

# Identifiers become fixed width text before anything else: the dictionary declares
# CNPJ as 14 characters and the consumer unit set as 5, and the integer form of both
# silently drops leading zeros.
normalizado = (bronze
    .withColumn("num_cnpj", F.lpad(F.col("NumCNPJ").cast("string"), 14, "0"))
    .withColumn("ide_conjunto", F.lpad(F.col("IdeConjUndConsumidoras").cast("string"), 5, "0"))
    .withColumn("ano", F.col("AnoIndice").cast("int"))
    .withColumn("mes", F.col("NumPeriodoIndice").cast("int"))
    .withColumn("sig_indicador", F.trim("SigIndicador"))
    .withColumn("dsc_conjunto", F.trim("DscConjUndConsumidoras"))
    .withColumn("sig_agente", F.trim("SigAgente"))
    .withColumn("valor", F.col("VlrIndiceEnviado").cast("double"))
    .withColumn("dat_geracao", F.col("DatGeracaoConjuntoDados"))
    .select("num_cnpj", "sig_agente", "ide_conjunto", "dsc_conjunto",
            "ano", "mes", "sig_indicador", "valor", "dat_geracao"))

linhas_bronze = normalizado.count()
unico = normalizado.dropDuplicates(CHAVE)
linhas_unicas = unico.count()

print(f"linhas na Bronze...........: {linhas_bronze:,}")
print(f"linhas apos deduplicacao...: {linhas_unicas:,}")
print(f"duplicatas removidas.......: {linhas_bronze - linhas_unicas:,}")

### 3.2 Layout largo e composição dos indicadores

O pivot transforma as 23 linhas de indicador de cada conjunto-mês em 23 colunas. A parcela ausente não produz nulo no formato longo: a linha simplesmente não existe, e por isso o preenchimento com zero é aplicado depois do giro.

Quatro indicadores são compostos a partir das parcelas:

- `dec` e `fec` seguem a composição normativa do PRODIST Módulo 8, com a regra condicionada ao ano: até 2021 incluem as parcelas externas, de 2022 em diante não
- `dec_fi` e `fec_fi` são o indicador próprio deste trabalho, limitado à interrupção de origem interna e não programada, incluindo ISE e Dia Crítico

O consolidado publicado pela ANEEL não entra como coluna do fato. Ele falta em parte da série e viraria campo nulo sem significado; a conferência contra ele fica na tabela de reconciliação.

In [ ]:
parcelas = sorted(r["sig_indicador"] for r in
                  unico.select("sig_indicador").distinct().collect())

largo = (unico
    .groupBy("num_cnpj", "ide_conjunto", "ano", "mes")
    .pivot("sig_indicador", parcelas)
    .agg(F.first("valor"))
    .na.fill(0.0))

# A missing parcel means no event of that kind, not a gap: INC is only reported on a
# Critical Day. The zero above is therefore a value, not a placeholder.


def soma(prefixo, sufixos):
    """Sum the parcel columns of one indicator, skipping the ones not published."""
    colunas = [f"{prefixo}{s}" for s in sufixos if f"{prefixo}{s}" in largo.columns]
    return sum(F.col(c) for c in colunas)


def composto(prefixo):
    """Normative indicator: the composition rule depends on the year."""
    return F.when(F.col("ano") < ANO_MUDANCA_REGIME,
                  soma(prefixo, COMPOSICAO["ate_2021"])
                  ).otherwise(soma(prefixo, COMPOSICAO["desde_2022"]))


fato = (largo
    .withColumn("regime", F.when(F.col("ano") < ANO_MUDANCA_REGIME,
                                 F.lit("ate_2021")).otherwise(F.lit("desde_2022")))
    .withColumn("dec", F.round(composto("DEC"), 4))
    .withColumn("fec", F.round(composto("FEC"), 4))
    .withColumn("dec_fi", F.round(soma("DEC", PARCELAS_FI), 4))
    .withColumn("fec_fi", F.round(soma("FEC", PARCELAS_FI), 4))
    .withColumnRenamed("NumCon", "num_con"))

print(f"parcelas encontradas: {len(parcelas)}")
print(f"grao conjunto x ano x mes: {fato.count():,} linhas")

### 3.3 Completude do ano civil

O acumulado de doze meses exige doze meses. Quando a distribuidora atrasa ou falha no envio, o acumulado fica menor sem que apareça nulo algum, e a comparação entre empresas passa a usar períodos diferentes.

A contagem é feita no nível da distribuidora, não do conjunto: o indicador que será comparado é o da empresa, e um conjunto isolado com mês faltante já se dilui na média ponderada. A flag não exclui linha — quem decide o expurgo é a Gold, e apenas no ano afetado, não na janela inteira.

In [ ]:
meses_dx = (fato
    .groupBy("num_cnpj", "ano")
    .agg(F.countDistinct("mes").alias("meses_no_ano")))

fato = (fato.join(meses_dx, ["num_cnpj", "ano"])
            .withColumn("ano_completo", F.col("meses_no_ano") == 12))

incompletos = (meses_dx
    .filter(F.col("meses_no_ano") < 12)
    .filter(F.col("ano").isin(ANOS_JANELA))
    .orderBy("ano", "num_cnpj"))

n_incompletos = incompletos.count()
print(f"distribuidora-ano incompleto na janela: {n_incompletos}")
if n_incompletos:
    display(incompletos)

## 4. `fato_continuidade_mensal`

Grão conjunto, ano e mês. Guarda as parcelas como publicadas, o peso `num_con` e os quatro indicadores compostos.

In [ ]:
COLUNAS_FATO = (["num_cnpj", "ide_conjunto", "ano", "mes", "regime"]
                + [c for c in fato.columns if c.startswith(("DEC", "FEC"))]
                + ["num_con", "dec", "fec", "dec_fi", "fec_fi",
                   "meses_no_ano", "ano_completo"])

fato_final = fato.select(*COLUNAS_FATO)

(fato_final.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{SILVER}.fato_continuidade_mensal"))

print(f"fato_continuidade_mensal: {fato_final.count():,} linhas, "
      f"{len(fato_final.columns)} colunas")

In [ ]:
# Unity Catalog comments are the project dictionary: written here, read by the script
# that generates md/dicionario_dados.md from information_schema.
COMENTARIOS_FATO = {
    "num_cnpj": "CNPJ da distribuidora, texto de 14 caracteres com zeros a esquerda",
    "ide_conjunto": "Codigo do conjunto de unidades consumidoras, texto de 5 caracteres",
    "ano": "Ano civil de apuracao do indicador",
    "mes": "Mes civil de apuracao, de 1 a 12",
    "regime": "Regime de composicao vigente no ano: ate_2021 ou desde_2022",
    "num_con": "Numero de unidades consumidoras do conjunto no mes; peso da media ponderada",
    "dec": "DEC composto pelas parcelas do regime vigente, conforme PRODIST Modulo 8",
    "fec": "FEC composto pelas parcelas do regime vigente, conforme PRODIST Modulo 8",
    "dec_fi": "DEC de falha interna: soma de IND, INE e INC; indicador proprio deste trabalho",
    "fec_fi": "FEC de falha interna: soma de IND, INE e INC; indicador proprio deste trabalho",
    "meses_no_ano": "Meses distintos enviados pela distribuidora no ano civil",
    "ano_completo": "Verdadeiro quando a distribuidora enviou os doze meses do ano",
}

spark.sql(f"""COMMENT ON TABLE {SILVER}.fato_continuidade_mensal IS
    'Indicadores coletivos de continuidade no grao conjunto-ano-mes, tipados,
     deduplicados e com DEC e FEC compostos a partir das parcelas. Preserva a serie
     inteira publicada pela ANEEL, sem filtro de janela nem de universo.'""")

for coluna, texto in COMENTARIOS_FATO.items():
    spark.sql(f"ALTER TABLE {SILVER}.fato_continuidade_mensal "
              f"ALTER COLUMN {coluna} COMMENT '{texto}'")

print(f"comentarios aplicados: {len(COMENTARIOS_FATO)} colunas")

## 5. `dim_conjunto`

Grão conjunto. O notebook 03 verificou que nenhum dos 3.980 códigos aparece com mais de uma descrição na série, o que dispensa critério de desempate para o nome.

O período de existência importa porque as distribuidoras transferem subestações entre conjuntos, geralmente em janeiro. Conjunto que entra no meio da série tem acumulado menor por não ter histórico, e não por ter melhorado.

In [ ]:
dim_conjunto = (unico
    .groupBy("ide_conjunto", "num_cnpj")
    .agg(F.max("dsc_conjunto").alias("dsc_conjunto"),
         F.min(F.concat_ws("-", F.col("ano"), F.lpad(F.col("mes"), 2, "0"))).alias("primeiro_mes"),
         F.max(F.concat_ws("-", F.col("ano"), F.lpad(F.col("mes"), 2, "0"))).alias("ultimo_mes"),
         F.countDistinct(F.concat_ws("-", F.col("ano"), F.col("mes"))).alias("meses_na_serie")))

(dim_conjunto.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{SILVER}.dim_conjunto"))

COMENTARIOS_CONJUNTO = {
    "ide_conjunto": "Codigo do conjunto de unidades consumidoras, texto de 5 caracteres",
    "num_cnpj": "CNPJ da distribuidora responsavel pelo conjunto",
    "dsc_conjunto": "Nome do conjunto conforme publicado pela ANEEL",
    "primeiro_mes": "Primeiro mes em que o conjunto aparece na serie, no formato AAAA-MM",
    "ultimo_mes": "Ultimo mes em que o conjunto aparece na serie, no formato AAAA-MM",
    "meses_na_serie": "Quantidade de meses em que o conjunto foi informado",
}

spark.sql(f"""COMMENT ON TABLE {SILVER}.dim_conjunto IS
    'Conjuntos de unidades consumidoras, com o nome publicado e o periodo de
     existencia na serie. O codigo e a identidade; o nome e apenas descritivo.'""")

for coluna, texto in COMENTARIOS_CONJUNTO.items():
    spark.sql(f"ALTER TABLE {SILVER}.dim_conjunto "
              f"ALTER COLUMN {coluna} COMMENT '{texto}'")

print(f"dim_conjunto: {dim_conjunto.count():,} conjuntos")

## 6. `dim_distribuidora`

Grão CNPJ. Carrega o critério de porte como atributo, não como filtro.

O porte vem da soma de `num_con` dos conjuntos da distribuidora, medida em dezembro do primeiro ano da janela. A média dos doze meses do mesmo ano entra ao lado, como sensibilidade: dezembro é um instante e pode cair num mês de reestruturação, enquanto a média dilui o efeito. Quando os dois critérios discordam sobre quem está acima do corte, dezembro prevalece e a empresa é marcada como fronteira.

O denominador é o da própria base. Nenhuma contagem de unidades consumidoras vinda de outra fonte entra aqui: as bases da ANEEL contam universos distintos, por critério e por erro de envio, e substituir um denominador pelo outro introduz erro maior que a divergência que se quer corrigir.

In [ ]:
ucs_mes = (fato_final
    .filter(F.col("ano") == ANO_REFERENCIA_PORTE)
    .groupBy("num_cnpj", "mes")
    .agg(F.sum("num_con").alias("ucs_no_mes")))

ucs_dezembro = (ucs_mes.filter(F.col("mes") == 12)
    .select("num_cnpj", F.col("ucs_no_mes").alias("qtd_ucs_dezembro")))

ucs_media = (ucs_mes
    .groupBy("num_cnpj")
    .agg(F.round(F.avg("ucs_no_mes"), 0).cast("long").alias("qtd_ucs_media")))

siglas = (unico.select("num_cnpj", "sig_agente").distinct()
    .groupBy("num_cnpj").agg(F.max("sig_agente").alias("sig_agente")))

dim_distribuidora = (siglas
    .join(ucs_dezembro, "num_cnpj", "left")
    .join(ucs_media, "num_cnpj", "left")
    .withColumn("grande_porte", F.coalesce(F.col("qtd_ucs_dezembro"), F.lit(0)) >= CORTE_UCS)
    .withColumn("grande_porte_media", F.coalesce(F.col("qtd_ucs_media"), F.lit(0)) >= CORTE_UCS)
    # Fronteira marks the companies the two criteria disagree about; December decides.
    .withColumn("fronteira", F.col("grande_porte") != F.col("grande_porte_media"))
    .withColumn("ano_referencia", F.lit(ANO_REFERENCIA_PORTE)))

(dim_distribuidora.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{SILVER}.dim_distribuidora"))

print(f"distribuidoras..............: {dim_distribuidora.count()}")
print(f"grande porte por dezembro...: {dim_distribuidora.filter('grande_porte').count()}")
print(f"grande porte pela media.....: {dim_distribuidora.filter('grande_porte_media').count()}")
print(f"na fronteira................: {dim_distribuidora.filter('fronteira').count()}")

display(dim_distribuidora
        .filter(F.col("grande_porte") | F.col("fronteira"))
        .select("sig_agente", "num_cnpj", "qtd_ucs_dezembro", "qtd_ucs_media",
                "grande_porte", "grande_porte_media", "fronteira")
        .orderBy(F.col("qtd_ucs_dezembro").desc()))

In [ ]:
COMENTARIOS_DX = {
    "num_cnpj": "CNPJ da distribuidora, texto de 14 caracteres com zeros a esquerda",
    "sig_agente": "Sigla da distribuidora conforme publicada pela ANEEL",
    "qtd_ucs_dezembro": "Soma de num_con dos conjuntos em dezembro do ano de referencia",
    "qtd_ucs_media": "Media mensal da soma de num_con no ano de referencia",
    "grande_porte": "Verdadeiro quando qtd_ucs_dezembro atinge o corte de 400 mil UCs",
    "grande_porte_media": "Mesmo corte aplicado a qtd_ucs_media, como sensibilidade",
    "fronteira": "Verdadeiro quando os dois criterios discordam sobre o porte",
    "ano_referencia": "Ano civil usado para medir o porte",
}

spark.sql(f"""COMMENT ON TABLE {SILVER}.dim_distribuidora IS
    'Distribuidoras com o criterio de porte registrado como atributo. O recorte do
     universo e aplicado na Gold pela flag grande_porte, nao por filtro nesta tabela.'""")

for coluna, texto in COMENTARIOS_DX.items():
    spark.sql(f"ALTER TABLE {SILVER}.dim_distribuidora "
              f"ALTER COLUMN {coluna} COMMENT '{texto}'")

print(f"comentarios aplicados: {len(COMENTARIOS_DX)} colunas")

## 7. `recon_continuidade`

O consolidado que a ANEEL publica está no próprio arquivo, como `SigIndicador` igual a `DEC` e `FEC`. A conferência é portanto interna ao dado e se reproduz a cada execução, sem consulta manual ao portal.

A tabela cobre apenas as combinações em que o publicado existe. Onde ele falta — 128 casos de `FEC` em 2025, entre outros — não há o que conferir, e a contagem do que ficou de fora é informada.

O papel da tabela é sustentar a decisão de compor sempre pelas parcelas. O notebook 03 encontrou 125 combinações divergentes, concentradas na ELEKTRO em junho de 2025, e a divergência é do consolidado, não da composição.

In [ ]:
publicado = (unico
    .filter(F.col("sig_indicador").isin(["DEC", "FEC"]))
    .select("num_cnpj", "ide_conjunto", "ano", "mes",
            F.col("sig_indicador").alias("indicador"),
            F.col("valor").alias("publicado")))

composto_longo = None
for nome in ("dec", "fec"):
    parte = (fato_final
        .select("num_cnpj", "ide_conjunto", "ano", "mes",
                F.lit(nome.upper()).alias("indicador"),
                F.col(nome).alias("composto")))
    composto_longo = parte if composto_longo is None else composto_longo.unionByName(parte)

recon = (publicado
    .join(composto_longo, ["num_cnpj", "ide_conjunto", "ano", "mes", "indicador"], "inner")
    .withColumn("diferenca", F.round(F.col("composto") - F.col("publicado"), 4))
    .withColumn("adere", F.abs(F.col("diferenca")) <= TOLERANCIA))

(recon.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{SILVER}.recon_continuidade"))

total_composto = composto_longo.count()
total_recon = recon.count()
divergentes = recon.filter(~F.col("adere")).count()

print(f"combinacoes compostas..............: {total_composto:,}")
print(f"combinacoes com publicado..........: {total_recon:,}")
print(f"sem publicado, fora da reconciliacao: {total_composto - total_recon:,}")
print(f"divergentes acima da tolerancia....: {divergentes:,}")

display(recon.filter(~F.col("adere"))
    .join(spark.table(f"{SILVER}.dim_distribuidora").select("num_cnpj", "sig_agente"), "num_cnpj")
    .groupBy("sig_agente", "ano", "mes", "indicador")
    .agg(F.count("*").alias("conjuntos"),
         F.round(F.max(F.abs(F.col("diferenca"))), 4).alias("maior_desvio"))
    .orderBy(F.col("conjuntos").desc()))

In [ ]:
COMENTARIOS_RECON = {
    "num_cnpj": "CNPJ da distribuidora, texto de 14 caracteres com zeros a esquerda",
    "ide_conjunto": "Codigo do conjunto de unidades consumidoras, texto de 5 caracteres",
    "ano": "Ano civil de apuracao",
    "mes": "Mes civil de apuracao, de 1 a 12",
    "indicador": "DEC ou FEC",
    "publicado": "Valor consolidado publicado pela ANEEL para o indicador",
    "composto": "Valor obtido pela soma das parcelas do regime vigente",
    "diferenca": "Composto menos publicado",
    "adere": "Verdadeiro quando a diferenca absoluta nao excede a tolerancia de arredondamento",
}

spark.sql(f"""COMMENT ON TABLE {SILVER}.recon_continuidade IS
    'Conferencia entre o indicador composto pelas parcelas e o consolidado publicado
     pela ANEEL, restrita as combinacoes em que o consolidado existe. Evidencia do
     criterio de qualidade, nao insumo de calculo.'""")

for coluna, texto in COMENTARIOS_RECON.items():
    spark.sql(f"ALTER TABLE {SILVER}.recon_continuidade "
              f"ALTER COLUMN {coluna} COMMENT '{texto}'")

print(f"comentarios aplicados: {len(COMENTARIOS_RECON)} colunas")

## 8. Validação da Silver

Se sujeira chegou aqui, o erro está nesta camada. Quatro verificações fecham a etapa: a chave é única, os identificadores têm o tamanho declarado, nenhum indicador é negativo e o total de conjuntos bate com a dimensão.

In [ ]:
fato_lido = spark.table(f"{SILVER}.fato_continuidade_mensal")
dim_cj = spark.table(f"{SILVER}.dim_conjunto")

testes = []

linhas = fato_lido.count()
unicas = fato_lido.select("num_cnpj", "ide_conjunto", "ano", "mes").distinct().count()
testes.append(("chave unica no grao",
               linhas == unicas,
               f"{linhas:,} linhas para {unicas:,} combinacoes distintas"))

larguras = (fato_lido
    .select(F.length("num_cnpj").alias("len_cnpj"),
            F.length("ide_conjunto").alias("len_conj"))
    .filter((F.col("len_cnpj") != 14) | (F.col("len_conj") != 5))
    .count())
testes.append(("identificadores com largura fixa",
               larguras == 0,
               f"{larguras:,} linhas fora de 14 e 5 caracteres"))

negativos = fato_lido.filter(
    (F.col("dec") < 0) | (F.col("fec") < 0) |
    (F.col("dec_fi") < 0) | (F.col("fec_fi") < 0) |
    (F.col("num_con") <= 0)).count()
testes.append(("indicadores e peso validos",
               negativos == 0,
               f"{negativos:,} linhas com indicador negativo ou peso nao positivo"))

cj_fato = fato_lido.select("ide_conjunto").distinct().count()
cj_dim = dim_cj.count()
testes.append(("conjuntos do fato presentes na dimensao",
               cj_fato == cj_dim,
               f"{cj_fato:,} no fato contra {cj_dim:,} na dimensao"))

for nome, passou, detalhe in testes:
    print(f"[{'OK' if passou else 'FALHOU':<7}] {nome:<40} {detalhe}")

if all(p for _, p, _ in testes):
    print("\nSilver de continuidade validada.")
else:
    print("\nHa teste sem passar; corrigir antes de seguir para a Gold.")

## Pendências documentadas

| Item | Situação | O que falta |
|---|---|---|
| Ano de referência do porte | Definido como o primeiro ano da janela | Confirmar se o corte usa 2022, depois da ampliação da janela, ou permanece em 2023 |
| Conjuntos que entram no meio da série | Identificados na `dim_conjunto` | Decidir na Gold se o conjunto sem histórico completo entra no acumulado da distribuidora |
| Expurgo por ano incompleto | Flag calculada, expurgo não aplicado | A Gold aplica o expurgo apenas no ano afetado |


## Autoavaliação desta etapa

A preencher após a execução.